This notebook populates a neo4j Graph Database with Pokemon information fetched from the pokeapi API (up to date until Gen 9), and allows for questions regarding the database to gpt-4o.

In [3]:
import requests
import json
from neo4j import GraphDatabase
from pathlib import Path
import os
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm

In [4]:
dotenv_path = Path('~/.env').expanduser()
load_dotenv(dotenv_path=dotenv_path)  # This line brings all environment variables from .env into os.environ

# Get variables
POKEDEX_URI = os.getenv('POKEDEX_URI')
POKEDEX_USER = os.getenv('POKEDEX_USER')
POKEDEX_PASSWORD = os.getenv('POKEDEX_PASSWORD')
#database_name = os.getenv('DATABASE_NAME')

driver = GraphDatabase.driver(POKEDEX_URI, auth=(POKEDEX_USER, POKEDEX_PASSWORD))
# if((driver.verify_connectivity())=='None'):
#     print("Driver connectivity okay")
driver.verify_connectivity()

------------------------------------------

In [5]:
# fetch pokemon data from a given URL
def fetch_pokemon_data(url):
    response = requests.get(url)
    return response.json()

base_url = "https://pokeapi.co/api/v2/pokemon" #base url

data = fetch_pokemon_data(base_url)

pokemon_list = []

# iterate through the pages and get data for each pokemon
while data:
    for pokemon in data['results']:
        pokemon_data = fetch_pokemon_data(pokemon['url'])

        # extract relevant information
        name = pokemon_data["name"]
        types = [t["type"]["name"] for t in pokemon_data["types"]]
        abilities = [a["ability"]["name"] for a in pokemon_data["abilities"]]

        # append to list
        pokemon_list.append({"name": name, "types": ", ".join(types), "abilities": ", ".join(abilities)})

    # get the next page (if any)
    next_url = data['next']
    if next_url:
        data = fetch_pokemon_data(next_url)
    else:
        break

# create collective df
df = pd.DataFrame(pokemon_list)
df['name'] = df['name'].str.capitalize()

print(df.head()) 

         name          types              abilities
0   Bulbasaur  grass, poison  overgrow, chlorophyll
1     Ivysaur  grass, poison  overgrow, chlorophyll
2    Venusaur  grass, poison  overgrow, chlorophyll
3  Charmander           fire     blaze, solar-power
4  Charmeleon           fire     blaze, solar-power


In [ ]:
# # Add an empty column to your DataFrame
# df['evolves_from'] = None

# # Loop through Pokémon and fetch evolution info
# for idx, row in tqdm(df.iterrows(), total=len(df)):
#     name = row['name'].lower()
#     url = f"https://pokeapi.co/api/v2/pokemon-species/{name}"
#     response = requests.get(url)

#     if response.status_code == 200:
#         species_data = response.json()
#         evolves_from = species_data.get('evolves_from_species')
#         if evolves_from:
#             df.at[idx, 'evolves_from'] = evolves_from['name'].capitalize()

100%|██████████| 1302/1302 [10:04<00:00,  2.15it/s]


In [16]:
df['types'] = df['types'].apply(lambda x: [t.strip().capitalize() for t in x.split(',')] if isinstance(x, str) else x)
df['abilities'] = df['abilities'].apply(lambda x: [t.strip().capitalize() for t in x.split(',')] if isinstance(x, str) else x)

def split_commas(list):
    result = []
    for t in list:
        if isinstance(t, str) and ',' in t:
            result.extend([s.strip().capitalize() for s in t.split(',')])
        else:
            result.append(t.strip().capitalize() if isinstance(t, str) else t)
    return result

df['types'] = df['types'].apply(split_commas)
df['abilities'] = df['abilities'].apply(split_commas)

print(df.head())

         name            types                abilities
0   Bulbasaur  [Grass, Poison]  [Overgrow, Chlorophyll]
1     Ivysaur  [Grass, Poison]  [Overgrow, Chlorophyll]
2    Venusaur  [Grass, Poison]  [Overgrow, Chlorophyll]
3  Charmander           [Fire]     [Blaze, Solar-power]
4  Charmeleon           [Fire]     [Blaze, Solar-power]


In [6]:
def write_pokemon(tx,statement, params_dict):
    tx.run(statement, parameters=params_dict)

def write_types(tx, statement, params_dict):
    tx.run(statement, parameters=params_dict)

def write_abilities(tx, statement, params_dict):
    tx.run(statement, parameters=params_dict)

In [6]:
statement_create_pokemon = """MERGE (p:Pokemon {name:$name})"""

with driver.session() as session:
    for index, row in df.iterrows():
        session.execute_write(write_pokemon, 
                                params_dict = {
                                    'name':str(row['name'])
                                },
                                statement = statement_create_pokemon)  

In [ ]:
statement_create_types = """MERGE (t:Type {name:$type})"""

with driver.session() as session:
    for index, row in df.iterrows():
        for type_name in row['types']:  # type_name will be 'water', 'grass', etc.
            session.execute_write(
                write_types, 
                params_dict={
                    'type': str(type_name).capitalize()  # capitalize type names
                },
                statement=statement_create_types
            )

In [18]:
statement_create_abilities = """MERGE (a:Ability {name:$ability})"""

with driver.session() as session:
    for index, row in df.iterrows():
        for ability_name in row['abilities']:  
            session.execute_write(
                write_abilities, 
                params_dict={
                    'ability': str(ability_name).capitalize() 
                },
                statement=statement_create_abilities
            )

In [10]:
statement_has_type = """MATCH (p:Pokemon {name: $name})
MATCH (t:Type {name:$type})
MERGE (p)-[:HAS_TYPE]->(t)
"""

with driver.session() as session:
    for index, row in df.iterrows():
        pokemon_name = row['name']
        for type_name in row['types']:
            session.execute_write(
                write_pokemon,
                params_dict = {
                    'name': pokemon_name,
                    'type': type_name
                },
                statement = statement_has_type
            )

C:\Users\Eleftheria\AppData\Local\Temp\ipykernel_19308\3795341116.py:6: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:


In [ ]:
statement_has_ability = """MATCH (p:Pokemon {name: $name})
MATCH (a:Ability {name:$ability})
MERGE (p)-[:HAS_ABILITY]->(a)
"""

with driver.session() as session:
    for index, row in df.iterrows():
        pokemon_name = row['name']
        for ability_name in row['abilities']:
            session.execute_write(
                write_pokemon,
                params_dict = {
                    'name': pokemon_name,
                    'ability': ability_name
                },
                statement = statement_has_ability
            )

---------------------------------------------------------- **LLM integration for question-answer rapport** ----------------------------------------------------------

In [ ]:
import openai

In [ ]:
openai.api_key = os.getenv('OPENAI_API_KEY')

client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def ask_gpt4o(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content


In [ ]:
def question_to_cypher_and_answer(question):

    # prompt LLM to translate the question into a Cypher query
    prompt = f"""
    You are a Cypher expert. Given the following question, respond ONLY with the Cypher query. 
    Do NOT include markdown or code block formatting. Just return the plain Cypher.

    Question: "{question}"
    """
    cypher_query = ask_gpt4o(prompt)
    print("Generated Cypher Query:\n", cypher_query)

    results = run_cypher(cypher_query)
    print("Query Results:\n", results)

    interpretation_prompt = f"""
    Given the question: "{question}"
    And the results: {results}
    List the results. Do not interpret them and do not check whether their are factually correct.
    """
    explanation = ask_gpt4o(interpretation_prompt)
    return explanation


In [8]:
question = "What are the types of Piplup?"
print(question_to_cypher_and_answer(question))

Generated Cypher Query:
 MATCH (p:Pokemon {name: 'Piplup'})-[:HAS_TYPE]->(t:Type)
RETURN t.name
Query Results:
 [{'t.name': 'Water'}]


C:\Users\Eleftheria\AppData\Local\Temp\ipykernel_16484\3894439219.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:


Water
